In [85]:
import sqlite3
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

conn = sqlite3.connect("../diabetes-readmission.db")
X_train = pd.read_sql(""" SELECT * FROM X_train""", conn)
X_test = pd.read_sql(""" SELECT * FROM X_test""", conn)
y_train = pd.read_sql(""" SELECT * FROM y_train""", conn)['readmitted']
y_test = pd.read_sql(""" SELECT * FROM y_test""", conn)['readmitted']
conn.close()

## Logistic Regression
The Logistic Regression baseline shows limited success in predicting patient readmission. The F1 score of 0.527 reflects poor performance in both precision and recall, and the AUC-PR of 0.535 sits only slightly above the baseline of approximately 0.41, which reflects the positive class prevalence in the dataset. The features engineered from clinical data do carry signal, but the complex interactions between diagnosis combinations, medication changes, and prior utilization history are unlikely to be captured by a linear decision boundary. A Random Forest classifier will be implemented next, with the goal of improving both precision and recall by modeling the non-linear relationships in the data.

In [86]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(class_weight='balanced', random_state=42).fit(X_train, y_train)

In [87]:
predictions = log_model.predict(X_test)
probs = log_model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, predictions, output_dict=True)
conf_matrix = confusion_matrix(y_test, predictions)
roc_score = roc_auc_score(y_test, probs)
pr_score = average_precision_score(y_test, probs)

print(f"CLASSIFICATION REPORT: {report['1']}")
print(f"CONFUSION MATRIX: {conf_matrix}")
print(f"AUC-ROC: {roc_score:.4f}")
print(f"AUC-PR: {pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.5027343253211243, 'recall': 0.5548070175438596, 'f1-score': 0.5274886575927409, 'support': 7125.0}
CONFUSION MATRIX: [[6459 3910]
 [3172 3953]]
AUC-ROC: 0.6294
AUC-PR: 0.5352


In [88]:
from sklearn.metrics import accuracy_score
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(f"Baseline (always predict 0): {y_test.value_counts(normalize=True)[0]:.4f}")

Accuracy: 0.5952
Baseline (always predict 0): 0.5927


In [89]:
results = pd.DataFrame([{
    'model': 'Logistic Regression',
    'recall': report['1']['recall'],
    'precision': report['1']['precision'],
    'f1': report['1']['f1-score'],
    'auc_roc': roc_score,
    'auc_pr': pr_score
}])

conn = sqlite3.connect('../diabetes-readmission.db')
results.to_sql('model_results', conn, if_exists='replace', index=False)
conn.close()

## Random Forest

In [90]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(n_estimators=100, class_weight='balanced',random_state=42, n_jobs=-1).fit(X_train, y_train)

In [91]:
predictions = forest_model.predict(X_test)
probs = forest_model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, predictions, output_dict=True)
conf_matrix = confusion_matrix(y_test, predictions)
roc_score = roc_auc_score(y_test, probs)
pr_score = average_precision_score(y_test, probs)

print(f"CLASSIFICATION REPORT: {report['1']}")
print(f"CONFUSION MATRIX: {conf_matrix}")
print(f"AUC-ROC: {roc_score:.4f}")
print(f"AUC-PR: {pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.5410536307546274, 'recall': 0.32, 'f1-score': 0.40215186524384866, 'support': 7125.0}
CONFUSION MATRIX: [[8435 1934]
 [4845 2280]]
AUC-ROC: 0.6138
AUC-PR: 0.5082


In [92]:
for threshold in [0.1, 0.15, 0.2, 0.25, 0.30, 0.35, 0.40, 0.45]:
    preds = (probs >= threshold).astype(int)
    report_t = classification_report(y_test, preds, output_dict=True)
    print(f"Threshold {threshold}: Precision={report_t['1']['precision']:.3f}, Recall={report_t['1']['recall']:.3f}, F1={report_t['1']['f1-score']:.3f}")

Threshold 0.1: Precision=0.409, Recall=0.998, F1=0.580
Threshold 0.15: Precision=0.412, Recall=0.990, F1=0.582
Threshold 0.2: Precision=0.418, Recall=0.970, F1=0.584
Threshold 0.25: Precision=0.428, Recall=0.931, F1=0.587
Threshold 0.3: Precision=0.440, Recall=0.860, F1=0.582
Threshold 0.35: Precision=0.459, Recall=0.759, F1=0.572
Threshold 0.4: Precision=0.481, Recall=0.627, F1=0.544
Threshold 0.45: Precision=0.505, Recall=0.479, F1=0.492


## XGBoost

In [93]:
from xgboost import XGBClassifier

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr'
).fit(X_train, y_train)


In [94]:
predictions = xgb_model.predict(X_test)
probs = xgb_model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, predictions, output_dict=True)
conf_matrix = confusion_matrix(y_test, predictions)
roc_score = roc_auc_score(y_test, probs)
pr_score = average_precision_score(y_test, probs)

print(f"CLASSIFICATION REPORT: {report['1']}")
print(f"CONFUSION MATRIX: {conf_matrix}")
print(f"AUC-ROC: {roc_score:.4f}")
print(f"AUC-PR: {pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.5092054263565892, 'recall': 0.5900350877192982, 'f1-score': 0.5466484623886614, 'support': 7125.0}
CONFUSION MATRIX: [[6317 4052]
 [2921 4204]]
AUC-ROC: 0.6413
AUC-PR: 0.5443


In [95]:
for threshold in [0.1, 0.15, 0.2, 0.25, 0.30, 0.35, 0.40, 0.45]:
    preds = (probs >= threshold).astype(int)
    report_t = classification_report(y_test, preds, output_dict=True)
    print(f"Threshold {threshold}: Precision={report_t['1']['precision']:.3f}, Recall={report_t['1']['recall']:.3f}, F1={report_t['1']['f1-score']:.3f}")

Threshold 0.1: Precision=0.407, Recall=1.000, F1=0.579
Threshold 0.15: Precision=0.408, Recall=0.999, F1=0.579
Threshold 0.2: Precision=0.409, Recall=0.997, F1=0.580
Threshold 0.25: Precision=0.414, Recall=0.990, F1=0.584
Threshold 0.3: Precision=0.423, Recall=0.967, F1=0.588
Threshold 0.35: Precision=0.437, Recall=0.919, F1=0.592
Threshold 0.4: Precision=0.454, Recall=0.847, F1=0.591
Threshold 0.45: Precision=0.478, Recall=0.745, F1=0.583


In [96]:
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7]:
    preds = (probs >= threshold).astype(int)
    report_t = classification_report(y_test, preds, output_dict=True)
    print(f"Threshold {threshold}: Precision={report_t['1']['precision']:.3f}, Recall={report_t['1']['recall']:.3f}, F1={report_t['1']['f1-score']:.3f}")

Threshold 0.5: Precision=0.509, Recall=0.590, F1=0.547
Threshold 0.55: Precision=0.554, Recall=0.400, F1=0.465
Threshold 0.6: Precision=0.592, Recall=0.268, F1=0.369
Threshold 0.65: Precision=0.630, Recall=0.170, F1=0.267
Threshold 0.7: Precision=0.680, Recall=0.099, F1=0.172
